# Chapter 03: Manipulator Kinematics

Source orientation: printed pp. 81-154; PDF pp. 99-172. Chapter question: **How do joint screws assemble into end-effector motion and force transmission?**

This notebook is an original, standalone teaching chapter. It uses the textbook for structure and mathematical orientation, but the explanations, code, and figures are created here. The chapter focus is product of exponentials, inverse kinematics, Jacobians, singularities, manipulability. The key objects are joint screws, home configurations, product-of-exponentials maps, inverse kinematics branches, spatial and body Jacobians, singular values, workspaces, and parallel constraints.

Generated artifacts live under `artifacts/chapter-03` and are displayed both by code and in the static gallery. The final sanity cell checks the mathematical claims and artifact files so that the notebook remains auditable after reruns.


## Translation Guide

The chapter reads a manipulator as a chain of exponentials. Each joint contributes a screw motion, and the ordered product carries the end-effector from the home pose to the current task pose. The computational translation used here is deliberately concrete: every named object becomes an array, graph, cone, trajectory, or map whose dimensions can be checked. That keeps the notebook independent from the PDF while preserving the chapter's mathematical route.

The main objects for this chapter are joint screws, home configurations, product-of-exponentials maps, inverse kinematics branches, spatial and body Jacobians, singular values, workspaces, and parallel constraints. Read each one as a map between spaces. Ask what frame or chart supplies its coordinates, what operation changes those coordinates, and what quantity should remain invariant. The visuals in this notebook are chosen to make those questions inspectable rather than decorative.

The source pages are used only as orientation. Definitions and examples are paraphrased into course language, and all figures are generated from fresh code. When a visual resembles a textbook concept, it is a redraw or computational experiment rather than a page crop.


## Route Through The Chapter

We move in four passes. First, the notebook names the chapter's geometric objects and their engineering purpose. Second, it generates the visual sequence: product of exponentials builder, workspace and manipulability, jacobian singularity gallery. Third, it runs a compact computational check that records the relevant residuals, ranks, endpoint errors, determinants, or signs. Fourth, it closes with an applied lab that asks the reader to change a parameter and explain what stayed invariant.

The important habit is to compare the visual artifact with the JSON check. If a cone claims feasibility, the feasibility check should agree. If a Jacobian claims usable motion, the task-space determinant or rank should agree. If a loop claims bracket displacement, the endpoint check should reveal it. The notebook is designed so those cross-checks are near each other.


In [ ]:
from pathlib import Path
import sys
import numpy as np

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the robotics course root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts"
TOPIC = "chapter-03"

from utils.artifacts import display_artifact, save_json
from utils.visuals import build_storyboard, storyboard_check_payload


In [ ]:
storyboard = {
  "label": "Chapter 03: Manipulator Kinematics",
  "artifact_topic": "chapter-03",
  "source_span": "printed pp. 81-154; PDF pp. 99-172",
  "visual_sequence": [
    {
      "kind": "manipulator",
      "concept": "Product Of Exponentials Builder",
      "filename": "product-of-exponentials-builder.png",
      "observation": "joint screws accumulate into a tool pose"
    },
    {
      "kind": "workspace",
      "concept": "Workspace And Manipulability",
      "filename": "workspace-and-manipulability.png",
      "observation": "reachable set and velocity ellipsoid"
    },
    {
      "kind": "singularity",
      "concept": "Jacobian Singularity Gallery",
      "filename": "jacobian-singularity-gallery.png",
      "observation": "rank loss as geometry collapses"
    }
  ]
}

visual_results = build_storyboard(storyboard, ARTIFACT_ROOT, TOPIC)
visual_payload = storyboard_check_payload(storyboard, visual_results)
for item in visual_results:
    display_artifact(item["path"], width=720)
visual_payload


## Static Artifact Gallery

![Product Of Exponentials Builder](../../artifacts/chapter-03/figures/product-of-exponentials-builder.png)

*Inspection target:* joint screws accumulate into a tool pose.

![Workspace And Manipulability](../../artifacts/chapter-03/figures/workspace-and-manipulability.png)

*Inspection target:* reachable set and velocity ellipsoid.

![Jacobian Singularity Gallery](../../artifacts/chapter-03/figures/jacobian-singularity-gallery.png)

*Inspection target:* rank loss as geometry collapses.


## Worked Interpretation

The planar arm check uses both SE(3) forward kinematics and a 2D endpoint Jacobian. The endpoint Jacobian measures actual tool velocity in the plane, so its determinant gives a meaningful manipulability margin away from singularity. This is the chapter's small worked example, not a full simulator. It is intentionally narrow enough that the relevant convention can be seen directly in code and broad enough to catch the common failure mode.

The visual sequence and the executable check use compatible parameters whenever possible. The point is to avoid a split course where the images tell one story and the numbers tell another. When extending this notebook, reuse that pattern: pick one invariant, draw the geometry that exposes it, then save a check payload that would fail if the convention were reversed or the rank condition were lost.

**Pitfall to watch:** Do not use the angular rows of a spatial Jacobian as a planar position Jacobian. The former tells how the rigid body frame twists; the latter tells how the endpoint moves in the task plane. Confusing them can make a healthy arm look motionless. This pitfall is why the notebook avoids silent coordinate conventions. Names, frames, dimensions, and signs are part of the teaching surface, not implementation trivia.


In [ ]:
from utils.robots import planar_2r, planar_position_jacobian, planar_workspace, manipulability

robot = planar_2r()
theta = np.array([0.55, -0.85])
T = robot.fkine(theta)
J_spatial = robot.spatial_jacobian(theta)
J_task = planar_position_jacobian(theta, (1.0, 0.75))
workspace = planar_workspace((1.0, 0.75), samples=28)
task_manipulability = manipulability(J_task)
check_payload = {
    "fkine_bottom_row_error": float(np.linalg.norm(T[3] - np.array([0, 0, 0, 1]))),
    "spatial_jacobian_shape": list(J_spatial.shape),
    "task_jacobian_shape": list(J_task.shape),
    "task_jacobian_rank": int(np.linalg.matrix_rank(J_task)),
    "workspace_samples": int(len(workspace)),
    "task_manipulability": float(task_manipulability),
}
assert check_payload["fkine_bottom_row_error"] < 1e-12
assert check_payload["task_jacobian_rank"] == 2
assert check_payload["task_manipulability"] > 0.1
check_payload


## Applied Lab

Lab prompt: Sample a planar arm workspace and compare its best-conditioned and worst-conditioned poses.

Before running the lab, make a prediction in three parts: which geometric object is being changed, which representation will show the change most clearly, and which scalar check should move in the same direction. After running it, compare the prediction with the saved JSON payload under `artifacts/chapter-03/labs`.

Use the pitfall above as a diagnostic. If the visual and scalar check disagree, inspect the frame convention, normalization, rank threshold, sign convention, or parameter range. The best result is not merely a green check; it is a short explanation of why the check protects the chapter's main idea.


In [ ]:
lab_notes = {
    "lab": 'Sample a planar arm workspace and compare its best-conditioned and worst-conditioned poses.',
    "source_orientation": "printed pp. 81-154; PDF pp. 99-172",
    "artifact_topic": TOPIC,
    "reader_task": "Change one parameter, rerun the visual cell, and explain which invariant did or did not change.",
}
lab_path = save_json(lab_notes, TOPIC, "labs", "applied-lab.json", root=ARTIFACT_ROOT)
display_artifact(lab_path)


In [ ]:
from pathlib import Path

visual_paths = [Path(item["path"]) for item in visual_results]
assert visual_payload["assertions"]["has_multiple_visuals"]
assert visual_payload["assertions"]["all_visuals_nonblank"]
assert all(path.exists() and path.stat().st_size > 200 for path in visual_paths)

final_sanity = {
    "visual_payload": visual_payload,
    "checks": check_payload,
    "visual_artifact_count": len(visual_paths),
    "visual_artifacts": [path.relative_to(BOOK_ROOT).as_posix() for path in visual_paths],
}
sanity_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json", root=ARTIFACT_ROOT)
display_artifact(sanity_path)
final_sanity


## Practice And Inspection Notes

Use this chapter as a small laboratory, not as a static summary. The source span is printed pp. 81-154 and PDF pp. 99-172, but the working material is the notebook itself: the generated artifacts, the executable check payload, and the applied lab. Start by reading the chapter question again: **How do joint screws assemble into end-effector motion and force transmission?** Then identify which part of the visual sequence gives the most direct answer. For this chapter the focus is product of exponentials, inverse kinematics, Jacobians, singularities, manipulability, so the useful inspection targets are not generic screenshots; they are the ranks, cones, motions, frames, phase loops, energy shapes, or dependency arrows that make that focus concrete.

The `product of exponentials builder` artifact uses the `manipulator` visual family; inspect it for joint screws accumulate into a tool pose. The `workspace and manipulability` artifact uses the `workspace` visual family; inspect it for reachable set and velocity ellipsoid. The `jacobian singularity gallery` artifact uses the `singularity` visual family; inspect it for rank loss as geometry collapses.

After viewing the artifacts, rerun the computational check cell and read the keys in `check_payload` as a miniature rubric. Each key should protect one concept from a plausible robotics mistake. A determinant or rank protects a singularity claim. A residual protects an equation or closure condition. A monotonicity flag protects a scale-law interpretation. An endpoint error protects a steering construction. A power-invariance error protects a frame transformation. If a check is near zero, ask why zero is the right target; if a check is positive, ask what physical or geometric margin it measures.

Try three variations. First, make a small parameter change that should preserve the chapter's main invariant, such as a coordinate-frame change, a harmless redraw scale, or a sample density increase. Second, make a parameter change that should stress the model, such as a near-singular joint pose, lower friction coefficient, larger controller delay, smaller bracket loop, or weaker tendon tension. Third, make a convention change deliberately, such as reversing a normal or swapping a body/spatial interpretation, and predict which check should fail. These variations turn the notebook from a solved example into a diagnostic tool.

When writing your own notes, use this sentence pattern: "The artifact shows ..., the computation checks ..., and the invariant is ... ." That pattern is intentionally repetitive because it catches vague understanding. A reader who can fill in those three blanks for this chapter can usually transfer the idea to a different mechanism, contact model, or control task without reopening the textbook.


## Takeaways

- The product of exponentials separates mechanism geometry from joint values.
- Jacobian columns are motion screws, and task-space Jacobians must match the task being measured.
- Singularities are geometric rank losses; the workspace picture and the numeric determinant should agree.
- Revisit the saved artifacts after changing parameters; the strongest learning usually comes from explaining why the invariant survived.
